## **LOCAL/ ON-PREMISE Large Language Model (LLM) Chatbot**

On‑Premise / Local LLM Chatbot is an AI‑powered conversational system deployed entirely within local infrastructure (e.g., personal computers, laboratory servers, or institutional data centers) without reliance on external cloud services. The chatbot utilizes a transformer‑based LLM to understand natural‑language input and generate human‑like responses while ensuring data privacy, security, and offline operation.

## **1. Setting Up the Development Environment**

Reference for creating Hugging Face Access Token HF_TOKEN

https://www.youtube.com/watch?v=uBSbgQ1qPHI  

https://www.youtube.com/watch?v=pB-j1VgP2s8

In [1]:
!pip install transformers torch gradio datasets

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import gradio as gr

In [3]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **2. Building a Baseline Chatbot**

✅ DialoGPT-medium is a Large Language Model (LLM)

✅ Use Decoder-only Transformer architecture

✅ Based on GPT‑2 (medium, ~345M parameters)

✅ Fine-tuned specifically for open‑domain conversation

✅ Designed to be run locally / on‑premise

In [4]:
# Load the pretrained DialoGPT model and tokenizer
MODEL_NAME = "microsoft/DialoGPT-medium"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [5]:
# Baseline chatbot function
chat_history_ids = None

def chatbot_response(user_input, chat_history_ids=None):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")
    # Add conversational history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids
    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response


## **3. Launch Your First Chatbot Locally**

In [6]:
css = """
/* Container */
.container {
    background-color: #fdf4f4;
    border-radius: 15px;
    box-shadow: 0 6px 20px rgba(0, 0, 0, 0.1);
    padding: 25px;
    font-family: 'Comic Sans MS', sans-serif;
}

/* Title */
h1 {
    text-align: center;
    font-size: 32px;
    color: #ff7f7f;
    font-weight: 600;
    margin-bottom: 25px;
    font-family: 'Pacifico', sans-serif;
}

/* Outer box */
.input_output_outerbox {
    background-color: #f8d3d3; /* Light pink */
    padding: 10px;
    border-radius: 15px;
    margin-bottom: 15px;
}

/* Input and Text area */
input[type="text"], textarea {
    width: 100%;
    padding: 18px 22px;
    font-size: 18px;
    border-radius: 25px;
    border: 2px solid #ff6f61;
    background-color: #fff9e6; /* Cream color */
    color: brown;
    font-weight: bold;
    outline: none;
    transition: border-color 0.3s ease;
}

/* Keep background and text color on focus */
input[type="text"]:focus, textarea:focus {
    border-color: #ff1493;
    background-color: #fff9e6 !important;
    color: brown;
    font-weight: bold;
    box-shadow: none;
}

/* Output */
.output_text {
    padding: 16px 22px;
    background-color: #2e082e;
    border-radius: 20px;
    font-size: 18px;
    color: brown;
    font-weight: bold;
    border: 1px solid #ff6f61;
    word-wrap: break-word;
    min-height: 60px;
}

/* Button */
button {
    background-color: #ff6f61;
    color: red;
    padding: 16px 28px;
    font-size: 20px;
    font-weight: bold;
    border-radius: 30px;
    border: none;
    cursor: pointer;
    width: 100%;
    transition: background-color 0.3s ease, transform 0.2s;
}

/* Button hover effect with animation */
button:hover {
    background-color: #ff1493;
    transform: scale(1.1);
}

/* Cute footer with smaller text */
footer {
    text-align: center;
    margin-top: 20px;
    font-size: 16px;
    color: #ff6f61;
}

"""


In [7]:
iface = gr.Interface(fn=chatbot_response,
                     theme="default",
                     inputs="text",
                     outputs="text",
                     title="Baseline Chatbot",
                     css=css)
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbf8f1fbaa0636583a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Resources adapted from https://www.linkedin.com/learning/instructors/zhongyu-pan?u=114913666